# 🥇 Step 3 — Aggregate to Gold

**Purpose:** Join the cleaned Silver tables and produce business-ready aggregate tables
in the `gold` schema. These Gold tables power dashboards, reports, and ad-hoc analytics.

### Gold Table Produced

| Gold Table | Description | Grain |
|---|---|---|
| `daily_revenue` | Total revenue and freight aggregated by calendar date | 1 row per day |

> **Prerequisites:** Run `02_transform_to_silver` first so the Silver tables exist.

---

In [0]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
# col      — column references
# to_date  — extract the date portion from a timestamp
# sum      — aggregate function for revenue calculation
# round    — round monetary values to 2 decimal places
from pyspark.sql.functions import col, to_date, sum, round

In [0]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
catalog = "medallion_trial"  # Must match the catalog used in previous notebooks

---
## 📊 Build the Daily Revenue Aggregate

**Logic:**
1. Read `silver.cleaned_orders` and `silver.cleaned_order_items`
2. Inner-join on `order_id` (only orders with line items)
3. Extract `order_date` from the purchase timestamp
4. Group by `order_date` and compute:
   - `total_revenue_usd` — sum of item prices
   - `total_freight_usd` — sum of freight/shipping costs
5. Write the result as `gold.daily_revenue`

In [0]:
print("Starting Gold Layer Aggregation...")

# ── 1. Read the cleaned Silver tables ─────────────────────────────────────
df_orders = spark.table(f"{catalog}.silver.cleaned_orders")
df_items  = spark.table(f"{catalog}.silver.cleaned_order_items")

# ── 2. Join orders with their line items ──────────────────────────────────
# Inner join ensures we only include orders that have at least one item.
df_joined = df_orders.join(df_items, on="order_id", how="inner")

# ── 3. Aggregate: daily revenue & freight ─────────────────────────────────
df_daily_revenue = (
    df_joined
    # Extract calendar date from the purchase timestamp
    .withColumn("order_date", to_date(col("order_purchase_timestamp")))
    .groupBy("order_date")
    .agg(
        round(sum("price"), 2).alias("total_revenue_usd"),
        round(sum("freight_value"), 2).alias("total_freight_usd"),
    )
    .orderBy("order_date")
)

# ── 4. Write to Gold as a managed Delta table ─────────────────────────────
(
    df_daily_revenue.write.format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.gold.daily_revenue")
)

print("✅ Daily Revenue table successfully saved to Gold!")

---
## 🔍 Validation Queries
Run the cells below to verify the Gold table was created correctly.

In [0]:
%sql
-- Retrieve the top 10 highest-revenue days (excluding nulls from missing timestamps)
SELECT *
FROM   medallion_trial.gold.daily_revenue
WHERE  order_date IS NOT NULL
ORDER  BY total_revenue_usd DESC
LIMIT  10;

In [0]:
%sql
-- Revenue trend over time — switch the Databricks result display to a line chart
-- to visualize the daily revenue progression.
SELECT
    order_date,
    total_revenue_usd
FROM   medallion_trial.gold.daily_revenue
ORDER  BY order_date ASC;

In [0]:
%sql
-- Dashboard widget: shows when this pipeline was last executed
SELECT date_format(current_timestamp(), 'MMM dd, yyyy - HH:mm:ss') AS Last_Refreshed;